# Notebook 5 — Preprocessing and Feature Pipeline
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

This notebook runs once. Everything downstream depends on its output.

It takes the parquet files saved by the four EDA notebooks, applies preprocessing,
tags linguistic subgroups, sets up the two evaluation tracks, creates the train and
test splits, and builds the FS1 feature matrix. The results are written to disk so
that every experiment notebook loads identical data.

**Why the order matters:** the vectoriser is fitted on the training partition only.
Fitting it on the full dataset before splitting would let test vocabulary leak into
the training feature space and inflate the apparent generalisation performance.

**Before running:** upload `thesis_utils.py` to  
`/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/`

**Cells:**
1. Install libraries and mount Drive
2. Import utilities and create folders
3. Load the EDA outputs
4. Preprocess text
5. Tag linguistic subgroups
6. Two-track label setup
7. Train / validation / test splits
8. Build FS1 feature matrix
9. Save everything
10. Verify

## Cell 1: Install and Mount

In [1]:
!pip install -q emoji scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 7.0 MB/s eta 0:00:00
Mounted at /content/drive
Drive mounted.


## Cell 2: Import Utilities

In [3]:
import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")

from thesis_utils import *
import pandas as pd, numpy as np

print()
setup_dirs()

thesis_utils loaded.
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal

  eda          -> /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Dataset_Exploration/EDA
  data         -> /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/data
  models       -> /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/models
  predictions  -> /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/predictions
  results      -> /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/results
  figures      -> /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/figures


## Cell 3: Load the EDA Outputs
Reads the parquet files written at the end of EDA Notebooks 1, 2 and 3.

In [4]:
EDA = PATHS["eda"]

df_tweeteval   = pd.read_parquet(EDA / "df_tweeteval.parquet")
df_semeval     = pd.read_parquet(EDA / "df_semeval.parquet")
df_s140_sample = pd.read_parquet(EDA / "df_s140_sample.parquet")

for name, d in [("TweetEval", df_tweeteval),
                ("SemEval-2014", df_semeval),
                ("Sentiment140 (100K sample)", df_s140_sample)]:
    print(f"{name:<28}: {len(d):>8,} rows")

print("\nTweetEval columns:", list(df_tweeteval.columns)[:8], "...")

TweetEval                   :   59,899 rows
SemEval-2014                :    7,694 rows
Sentiment140 (100K sample)  :  100,000 rows

TweetEval columns: ['text', 'label', 'split', 'sentiment', 'char_count', 'word_count', 'emoji_count', 'has_emoji'] ...


## Cell 4: Preprocess Text

Six steps: lowercase, strip URLs, strip @mentions, collapse repeated punctuation,
normalise whitespace, and for Sentiment140 only, strip emoticons.

The emoticon step is specific to Sentiment140 because its labels were derived from
emoticons. The EDA found 1,444 still present in the text. Leaving them in would let
any model read the label directly rather than learning sentiment from words.

Stemming and lemmatisation are deliberately excluded: 'amazingly' and 'amazing'
carry different sarcasm signals and collapsing them destroys a needed feature.

In [5]:
df_tweeteval = preprocess_column(df_tweeteval, "text", strip_emoticons=False)
df_semeval   = preprocess_column(df_semeval,   "text", strip_emoticons=False)

# Sentiment140 only — emoticon leakage fix
before = df_s140_sample["text"].str.contains(EMOTICON_PATTERN, regex=True).sum()
df_s140_sample = preprocess_column(df_s140_sample, "text", strip_emoticons=True)
after  = df_s140_sample["text_clean"].str.contains(EMOTICON_PATTERN, regex=True).sum()

print(f"Sentiment140 emoticon leakage: {before:,} before  ->  {after:,} after")
assert after == 0, "Emoticons still present. Do not proceed."

print("\nSample before / after:")
for i in range(2):
    print(f"  raw   : {df_tweeteval['text'].iloc[i][:90]}")
    print(f"  clean : {df_tweeteval['text_clean'].iloc[i][:90]}")
    print()

Sentiment140 emoticon leakage: 6,562 before  ->  0 after

Sample before / after:
  raw   : "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwar
  clean : "qt in the original draft of the 7th book, remus lupin survived the battle of hogwarts. #h

  raw   : "Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"
  clean : "ben smith / smith (concussion) remains out of the lineup thursday, curtis #nhl #sj"



## Cell 5: Tag Linguistic Subgroups

Rules are defined once in `thesis_utils.py` and applied identically to all three
datasets. If a post matches more than one rule the priority is
sarcasm, then emoji-heavy, then slang-heavy, then formal.

Subgroup labels are evaluation metadata. They are never passed to a model as a
training feature.

In [6]:
print("Tagging subgroups (uses raw text, not cleaned — emoji are stripped by cleaning)\n")

df_tweeteval   = tag_subgroups(df_tweeteval,   text_col="text")
df_semeval     = tag_subgroups(df_semeval,     text_col="text")
df_s140_sample = tag_subgroups(df_s140_sample, text_col="text")

for name, d in [("TweetEval", df_tweeteval),
                ("SemEval-2014", df_semeval),
                ("Sentiment140", df_s140_sample)]:
    print(f"{name}:")
    counts = d["subgroup_primary"].value_counts()
    for sg, n in counts.items():
        print(f"    {sg:<14}: {n:>7,}  ({n/len(d)*100:>5.1f}%)")
    print()

print("Cross-check against EDA — TweetEval should be roughly:")
print("  formal 57,307 | emoji-heavy 696 | slang-heavy 270 | sarcasm 112")

Tagging subgroups (uses raw text, not cleaned — emoji are stripped by cleaning)

TweetEval:
    formal        :  57,307  ( 95.7%)
    other         :   1,514  (  2.5%)
    emoji-heavy   :     696  (  1.2%)
    slang-heavy   :     270  (  0.5%)
    sarcasm       :     112  (  0.2%)

SemEval-2014:
    formal        :   6,869  ( 89.3%)
    other         :     793  ( 10.3%)
    sarcasm       :      26  (  0.3%)
    slang-heavy   :       6  (  0.1%)

Sentiment140:
    formal        :  72,742  ( 72.7%)
    other         :  24,301  ( 24.3%)
    slang-heavy   :   2,730  (  2.7%)
    sarcasm       :     167  (  0.2%)
    emoji-heavy   :      60  (  0.1%)

Cross-check against EDA — TweetEval should be roughly:
  formal 57,307 | emoji-heavy 696 | slang-heavy 270 | sarcasm 112


## Cell 6: Two-Track Label Setup

Track A keeps three classes and covers TweetEval and SemEval.
Track B is binary and covers all three datasets, with neutral rows dropped
rather than relabelled.

Sentiment140 has no neutral class, which is why the two tracks exist at all.

In [8]:
def setup_tracks(df, name):
    df = df.copy()
    df["sentiment_3class"] = df["sentiment"]
    df["sentiment_binary"] = df["sentiment"].where(df["sentiment"] != "neutral")
    df["has_neutral"]      = df["sentiment"] == "neutral"
    n_a = df["sentiment_3class"].notna().sum()
    n_b = df["sentiment_binary"].notna().sum()
    print(f"{name:<16}: Track A = {n_a:>8,}   Track B = {n_b:>9,}")
    return df

df_tweeteval   = setup_tracks(df_tweeteval,   "TweetEval")
df_semeval     = setup_tracks(df_semeval,     "SemEval-2014")
df_s140_sample = setup_tracks(df_s140_sample, "Sentiment140")

print("\nExpected from EDA: TweetEval A=59,899 B=32,420 | SemEval A=7,694 B=6,228")

TweetEval       : Track A =   59,899   Track B =    32,420
SemEval-2014    : Track A =    7,694   Track B =     6,228
Sentiment140    : Track A =  100,000   Track B =   100,000

Expected from EDA: TweetEval A=59,899 B=32,420 | SemEval A=7,694 B=6,228


## Cell 7: Train / Validation / Test Splits

TweetEval and SemEval have native splits, which are used as provided so that
results stay comparable with published benchmarks. Sentiment140 gets a stratified
80/10/10 split from the 100K development sample.

Seed 42 throughout.

In [9]:
from sklearn.model_selection import train_test_split

def make_splits(df, name, use_native=True):
    if use_native and "split" in df.columns:
        tr = df[df["split"] == "train"].copy()
        te = df[df["split"] == "test"].copy()
        va = df[df["split"] == "validation"].copy()
        if len(va) == 0:                       # SemEval has no validation split
            va = None
    else:
        tr, tmp = train_test_split(df, test_size=0.2,
                                   stratify=df["sentiment"], random_state=SEED)
        va, te  = train_test_split(tmp, test_size=0.5,
                                   stratify=tmp["sentiment"], random_state=SEED)
    n_va = len(va) if va is not None else 0
    print(f"{name:<16}: train {len(tr):>7,} | val {n_va:>6,} | test {len(te):>7,}")
    return tr, va, te

tw_train, tw_val, tw_test = make_splits(df_tweeteval,   "TweetEval")
se_train, se_val, se_test = make_splits(df_semeval,     "SemEval-2014")
s140_train, s140_val, s140_test = make_splits(df_s140_sample, "Sentiment140",
                                              use_native=False)

print("\nTest set subgroup distribution (TweetEval) — this is what fairness is measured on:")
print(tw_test["subgroup_primary"].value_counts())

TweetEval       : train  45,615 | val  2,000 | test  12,284
SemEval-2014    : train   5,936 | val      0 | test   1,758
Sentiment140    : train  80,000 | val 10,000 | test  10,000

Test set subgroup distribution (TweetEval) — this is what fairness is measured on:
subgroup_primary
formal         10398
other           1116
emoji-heavy      696
slang-heavy       60
sarcasm           14
Name: count, dtype: int64


## Cell 8: Build FS1 Feature Matrix

FS1 is TF-IDF with unigrams and bigrams, min_df 2, max_df 0.95, capped at 50,000
features. It is the baseline configuration for Experiments 1 to 6.

The vectoriser is fitted on the training partition only, then applied to validation
and test. This is the data leakage prevention step.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

def build_fs1(train_df, val_df, test_df, name):
    vec = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=50_000,
        sublinear_tf=True,
    )
    X_train = vec.fit_transform(train_df["text_clean"])      # FIT on train only
    X_val   = vec.transform(val_df["text_clean"]) if val_df is not None else None
    X_test  = vec.transform(test_df["text_clean"])           # transform only

    print(f"{name:<16}: vocab {len(vec.vocabulary_):>7,} | "
          f"train {X_train.shape} | test {X_test.shape}")
    return vec, X_train, X_val, X_test

print("Building FS1 (TF-IDF unigram + bigram)\n")
tw_vec, tw_Xtr, tw_Xva, tw_Xte = build_fs1(tw_train, tw_val, tw_test, "TweetEval")
se_vec, se_Xtr, se_Xva, se_Xte = build_fs1(se_train, se_val, se_test, "SemEval-2014")

y_tw_train = tw_train["sentiment"].values
y_tw_val   = tw_val["sentiment"].values if tw_val is not None else None
y_tw_test  = tw_test["sentiment"].values
sg_tw_test = tw_test["subgroup_primary"].values

print("\nLabel distribution in TweetEval training set:")
print(pd.Series(y_tw_train).value_counts())

Building FS1 (TF-IDF unigram + bigram)

TweetEval       : vocab  50,000 | train (45615, 50000) | test (12284, 50000)
SemEval-2014    : vocab  22,424 | train (5936, 22424) | test (1758, 22424)

Label distribution in TweetEval training set:
neutral     20673
positive    17849
negative     7093
Name: count, dtype: int64


## Cell 9: Save Everything

Saves the processed dataframes, the fitted vectoriser, and the feature matrices.
Every experiment notebook loads from here, so the preprocessing runs once only.

In [11]:
import pickle, scipy.sparse as sp

D = PATHS["data"]

# processed dataframes
df_tweeteval.to_parquet(D / "tweeteval_processed.parquet", index=False)
df_semeval.to_parquet(D / "semeval_processed.parquet", index=False)
df_s140_sample.to_parquet(D / "s140_processed.parquet", index=False)

# splits
for name, d in [("tw_train", tw_train), ("tw_val", tw_val), ("tw_test", tw_test),
                ("se_train", se_train), ("se_test", se_test),
                ("s140_train", s140_train), ("s140_test", s140_test)]:
    if d is not None:
        d.to_parquet(D / f"{name}.parquet", index=False)

# vectorisers
with open(D / "tw_vectorizer_fs1.pkl", "wb") as f: pickle.dump(tw_vec, f)
with open(D / "se_vectorizer_fs1.pkl", "wb") as f: pickle.dump(se_vec, f)

# sparse feature matrices
sp.save_npz(D / "tw_Xtrain_fs1.npz", tw_Xtr)
sp.save_npz(D / "tw_Xtest_fs1.npz",  tw_Xte)
if tw_Xva is not None:
    sp.save_npz(D / "tw_Xval_fs1.npz", tw_Xva)
sp.save_npz(D / "se_Xtrain_fs1.npz", se_Xtr)
sp.save_npz(D / "se_Xtest_fs1.npz",  se_Xte)

print("Saved to", D)
for f in sorted(D.glob("*")):
    print(f"  {f.name}")

Saved to /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/data
  s140_processed.parquet
  s140_test.parquet
  s140_train.parquet
  se_Xtest_fs1.npz
  se_Xtrain_fs1.npz
  se_test.parquet
  se_train.parquet
  se_vectorizer_fs1.pkl
  semeval_processed.parquet
  tw_Xtest_fs1.npz
  tw_Xtrain_fs1.npz
  tw_Xval_fs1.npz
  tw_test.parquet
  tw_train.parquet
  tw_val.parquet
  tw_vectorizer_fs1.pkl
  tweeteval_processed.parquet


## Cell 10: Verify

In [12]:
print("="*60)
print("NOTEBOOK 5 COMPLETE")
print("="*60)
print(f"TweetEval train  : {tw_Xtr.shape[0]:>7,} rows x {tw_Xtr.shape[1]:,} features")
print(f"TweetEval test   : {tw_Xte.shape[0]:>7,} rows")
print(f"SemEval train    : {se_Xtr.shape[0]:>7,} rows")
print(f"SemEval test     : {se_Xte.shape[0]:>7,} rows")
print()
print("Test set subgroup sizes (fairness metrics computed on these):")
for sg, n in tw_test["subgroup_primary"].value_counts().items():
    flag = "  <-- small, use bootstrap CI" if n < 100 else ""
    print(f"  {sg:<14}: {n:>6,}{flag}")
print()
print("Next: Notebook 6 — Experiments 1 to 6 (baselines)")

NOTEBOOK 5 COMPLETE
TweetEval train  :  45,615 rows x 50,000 features
TweetEval test   :  12,284 rows
SemEval train    :   5,936 rows
SemEval test     :   1,758 rows

Test set subgroup sizes (fairness metrics computed on these):
  formal        : 10,398
  other         :  1,116
  emoji-heavy   :    696
  slang-heavy   :     60  <-- small, use bootstrap CI
  sarcasm       :     14  <-- small, use bootstrap CI

Next: Notebook 6 — Experiments 1 to 6 (baselines)
